In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis: LBCO, HRPT

This tutorial demonstrates a practical two-stage workflow for powder
diffraction analysis with EasyDiffraction.

In the first stage, we run a fast local refinement to obtain a sensible
point estimate and parameter uncertainties. In the second stage, we use
these refined values to define fit bounds and then sample the posterior
distribution with DREAM.

The example uses constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 measured on HRPT at PSI.

The goal is not only to obtain a good fit, but also to answer Bayesian
questions such as:

- Which parameter values are most probable?
- How broad are the credible intervals?
- Which parameters are strongly correlated?
- How much uncertainty propagates into the calculated diffraction
  pattern?

## Import Library

In [2]:
import easydiffraction as ed

## Step 1: Create a Project Container

The project object keeps structures, experiments, fit settings, and
plotting utilities together in a single place. We will build the full
workflow inside this object.

In [3]:
project = ed.Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 2: Build the Structural Model

We define a simple cubic perovskite model for LBCO. La and Ba share the
same crystallographic site with equal occupancy, while Co and O occupy
the remaining ideal perovskite positions.

In [4]:
project.structures.create(name='lbco')

In [5]:
structure = project.structures['lbco']

In [6]:
structure.space_group.name_h_m = 'P m -3 m'
structure.space_group.it_coordinate_system_code = '1'

In [7]:
structure.cell.length_a = 3.88

The atom-site definitions below form the starting structural model. The
parameters are intentionally reasonable rather than fully optimized,
because the refinement step will improve them.

In [8]:
structure.atom_sites.create(
    label='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_type='Biso',
    adp_iso=0.5151,
    occupancy=0.5,
)
structure.atom_sites.create(
    label='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_type='Biso',
    adp_iso=0.2190,
)
structure.atom_sites.create(
    label='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_type='Biso',
    adp_iso=1.3916,
)

## Step 3: Define the Diffraction Experiment

Next we download the measured powder pattern, create a neutron powder
experiment, and configure the instrument, profile, background, and
excluded regions.

Download the measured data from the repository. Alternatively, you
could use your own data file by providing the path to it instead of
downloading from the repository.

In [9]:
data_path = ed.download_data(id=3, destination='data')

Getting data...


Data #3: La0.5Ba0.5CoO3, HRPT (PSI), 300 K


✅ Data #3 already present at 'data/ed-3.xye'. Keeping existing file.


Create the experiment object and specify the sample form, beam mode,
and radiation probe.

In [10]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully


Experiment 🔬 'hrpt'. Number of data points: 3098.


In [11]:
experiment = project.experiments['hrpt']

Link the structural phase to the experiment.

In [12]:
experiment.linked_phases.create(id='lbco', scale=9.1351)

Set instrument and peak profile parameters.

These values provide the initial instrument description for the local
refinement. Later, a subset of them will be refined.

In [13]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.0

In [14]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1204
experiment.peak.broad_lorentz_y = 0.0844

Add background points and excluded regions.

The line-segment background is defined by a few anchor points. We also
exclude regions that are not intended to contribute to the fit.

In [15]:
experiment.background.create(id='1', x=10, y=168.5585)
experiment.background.create(id='2', x=30, y=164.3357)
experiment.background.create(id='3', x=50, y=166.8881)
experiment.background.create(id='4', x=110, y=175.4006)

In [16]:
experiment.excluded_regions.create(id='1', start=0, end=10)
experiment.excluded_regions.create(id='2', start=100, end=180)

## Step 4: Run an Initial Local Refinement

Before Bayesian sampling, it is useful to run a deterministic fit. This
gives us:

- a good point estimate near the best-fit region,
- uncertainties from the local optimizer,
- a quick check that the model and experiment are configured
  sensibly.

In this tutorial we refine only a small set of parameters that are easy
to interpret in the later Bayesian stage.

In [17]:
structure.cell.length_a.free = True

In [18]:
experiment.linked_phases['lbco'].scale.free = True
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.instrument.calib_twotheta_offset.free = True

We choose the BUMPS Levenberg-Marquardt minimizer as a fast local
optimizer. Its main purpose here is to provide a stable starting point
and uncertainty estimates for the Bayesian run.

In [19]:
project.analysis.fit.show_minimizer_types()

Minimizer types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,bumps,Bumps library using the default Levenberg-Marquardt method
2,,bumps (amoeba),Bumps library with Nelder-Mead simplex method
3,,bumps (de),Bumps library with differential evolution method
4,,bumps (dream),Bumps library with DREAM Bayesian sampling
5,,bumps (lm),Bumps library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt least squares method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [20]:
project.analysis.fit.minimizer_type = 'bumps (lm)'

Current minimizer changed to


bumps (lm)


In [21]:
project.analysis.fit()

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.58,377.51,
2,8,1.45,56.31,85.1% ↓
3,14,2.17,39.54,29.8% ↓
4,21,2.85,37.66,4.7% ↓
5,26,3.62,34.14,9.4% ↓
6,27,3.96,23.47,31.3% ↓
7,33,4.66,8.74,62.7% ↓
8,39,5.48,1.85,78.9% ↓
9,45,6.35,1.30,29.9% ↓
10,77,8.78,1.29,


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🏆 Best goodness-of-fit (reduced χ²) is 1.29 at iteration 76


✅ Fitting complete.


In [22]:
project.analysis.display.fit_results()

Fit results


✅ Success: True


⏱️ Fitting time: 8.78 seconds


📏 Goodness-of-fit (reduced χ²): 1.29


📏 R-factor (Rf): 5.65%


...

📏 Weighted R-factor (wR): 4.08%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,start,fitted,uncertainty,change
1,lbco,cell,,length_a,Å,3.8800,3.8913,0.0001,0.29 % ↑
2,hrpt,linked_phases,lbco,scale,,9.1351,9.1329,0.0333,0.02 % ↓
3,hrpt,peak,,broad_gauss_u,deg²,0.1000,0.0817,0.0078,18.33 % ↓
4,hrpt,peak,,broad_gauss_v,deg²,-0.1000,-0.1169,0.0057,16.91 % ↑
5,hrpt,instrument,,twotheta_offset,deg,0.0000,0.6306,0.0019,N/A


The correlation plot shows how strongly the fitted parameters move
together in the local refinement. The measured-vs-calculated plots show
how well the refined model reproduces the data globally and in a zoomed
region.

In [23]:
project.display.plotter.plot_param_correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
project.display.plotter.plot_meas_vs_calc(expt_name='hrpt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 5: Prepare for Bayesian Sampling

DREAM requires finite bounds for the free parameters. Instead of
setting them manually, we derive them from the uncertainties estimated
in the local refinement.

The helper method `set_fit_bounds_from_uncertainty` centers the bounds
on the current parameter value and expands them by a chosen multiple of
the reported uncertainty.

The default `multiplier` is 4. If the local refinement is very tight,
or if you expect a broader posterior, increase it explicitly.

Show unset fit bounds before setting them from the local refinement uncertainties.

In [25]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89134,0.00011,-inf,inf,Å
2,hrpt,linked_phases,lbco,scale,9.13288,0.03329,-inf,inf,
3,hrpt,peak,,broad_gauss_u,0.08167,0.00783,-inf,inf,deg²
4,hrpt,peak,,broad_gauss_v,-0.11691,0.00566,-inf,inf,deg²
5,hrpt,instrument,,twotheta_offset,0.63057,0.00191,-inf,inf,deg


Set fit bounds for all free parameters using the default multiplier of
4. In this tutorial that means the posterior pair plot will later
refer to a `±4 × uncertainty` region in its title. To use a different
region, pass another value, for example `multiplier=6`.

In [26]:
for param in project.free_parameters:
    param.set_fit_bounds_from_uncertainty()

Displaying the free parameters again is a convenient way to confirm
that the fit bounds have been assigned as expected before launching the
sampler.

In [27]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lbco,cell,,length_a,3.89134,0.00011,3.89091,3.89177,Å
2,hrpt,linked_phases,lbco,scale,9.13288,0.03329,8.99973,9.26603,
3,hrpt,peak,,broad_gauss_u,0.08167,0.00783,0.05034,0.11300,deg²
4,hrpt,peak,,broad_gauss_v,-0.11691,0.00566,-0.13955,-0.09427,deg²
5,hrpt,instrument,,twotheta_offset,0.63057,0.00191,0.62292,0.63823,deg


## Step 6: Configure and Run DREAM

We now switch from the local minimizer to the Bayesian DREAM sampler.

The settings below are intentionally small so the tutorial runs
quickly. For production analysis you would usually increase the number
of steps (`steps`) and often the burn-in (`burn`) as well. When
needed, the DREAM API also lets you tune how chains are initialized
through the `init` setting. Other sampler settings such as `thin` and
`pop` can be adjusted as well. The current EasyDiffraction defaults
use `steps=3000`, `init='lhs'`, and `parallel=0`, which tells
BUMPS-DREAM to use all available CPUs for population evaluations.

The `burn` setting is auto-resolved when left unset. With the default
`steps=3000` this gives `burn=600`, but if you override `steps` and
keep `burn=None`, the effective burn-in is recomputed automatically.
Here we use a much smaller step count to keep the tutorial fast, but
this is not recommended for production analysis.

In [28]:
project.analysis.fit.show_minimizer_types()

Minimizer types


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,,Type,Description
1,,bumps,Bumps library using the default Levenberg-Marquardt method
2,,bumps (amoeba),Bumps library with Nelder-Mead simplex method
3,,bumps (de),Bumps library with differential evolution method
4,,bumps (dream),Bumps library with DREAM Bayesian sampling
5,*,bumps (lm),Bumps library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,lmfit,LMFIT library using the default Levenberg-Marquardt least squares method
8,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
9,,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [29]:
project.analysis.fit.minimizer_type = 'bumps (dream)'

Current minimizer changed to


bumps (dream)


In [30]:
project.analysis.fit.minimizer.steps = 300  # lower than the default 3000

In [31]:
project.analysis.fit()

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'bumps (dream)'...


📈 Bayesian sampling progress:


,iteration,progress,time (s),log posterior,phase
1,1/361,0.3%,30.67,-1357.16,burn-in
2,21/361,5.8%,47.71,-1166.88,burn-in
3,40/361,11.1%,66.93,-1161.19,burn-in
4,60/361,16.6%,86.85,-1160.38,burn-in
5,61/361,16.9%,88.05,-1160.07,sampling
6,76/361,21.1%,102.51,-1159.57,sampling
7,91/361,25.2%,116.85,-1159.38,sampling
8,106/361,29.4%,122.57,-1159.11,sampling
9,121/361,33.5%,128.16,-1158.94,sampling
10,136/361,37.7%,134.06,-1159.22,sampling


...

<IPython.core.display.Javascript object>

⚠️ Convergence diagnostics indicate the posterior may be poorly mixed.                                                            


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Bayesian sampling complete.


## Step 7: Inspect Bayesian Results

The fit-results display now includes sampler settings, convergence
diagnostics, committed parameter values, and posterior summary
statistics.

In [32]:
project.analysis.display.fit_results()

Bayesian fit results


⚠️ Overall status: completed with warnings


💬 Sampler status: DREAM sampling completed


🧪 Sampler: dream


🎯 Committed point estimate: Max posterior


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,median,95% interval,r-hat,ess bulk
1,lbco,cell,,length_a,Å,3.8913,"[3.8911, 3.8915]",1.091,167.0
2,hrpt,linked_phases,lbco,scale,,9.1322,"[9.0736, 9.1926]",1.072,276.5
3,hrpt,peak,,broad_gauss_u,deg²,0.0814,"[0.0683, 0.0937]",1.117,153.7
4,hrpt,peak,,broad_gauss_v,deg²,-0.1170,"[-0.1256, -0.1072]",1.113,154.9
5,hrpt,instrument,,twotheta_offset,deg,0.6301,"[0.6267, 0.6335]",1.089,165.2


⚠️ r-hat > 1.01: Consider longer sampling, better initialization, or reparameterization.                                          


⚠️ ess bulk < 400: Consider longer sampling or reparameterization.                                                                


The correlation and posterior-pair plots are complementary:

- `plot_param_correlations` summarizes pairwise structure in a compact
  matrix.
- `plot_posterior_pairs` shows marginal densities on the diagonal and
  posterior contours off-diagonal. In this tutorial its title also
  reminds you that the display region follows the `±4 × uncertainty`
  bounds defined above, while numeric subplot ranges are omitted to
  keep the grid readable.

In [33]:
project.display.plotter.plot_param_correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
project.display.plotter.plot_posterior_pairs()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The one-dimensional posterior distributions below make it easier to
inspect individual parameters in isolation, including asymmetry or
multimodality.

In [35]:
for param in project.free_parameters:
    project.display.plotter.plot_param_distribution(param)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Finally, the posterior predictive plot propagates the sampled parameter
uncertainty into the calculated diffraction pattern. Comparing this to
the zoomed measured-vs-calculated view helps assess whether the sampled
model family explains the data in the region of interest.

In [36]:
project.display.plotter.plot_posterior_predictive(expt_name='hrpt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

A final zoomed measured-vs-calculated plot is useful for checking how
the posterior-supported model behaves in a narrow region of the pattern
after the Bayesian run.

In [37]:
project.display.plotter.plot_posterior_predictive(
    expt_name='hrpt',
    x_min=92,
    x_max=93,
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>